# MODELO SARIMA IPCA

In [3]:
def teste_adf(serie):
    """
    Realiza o teste Augmented Dickey-Fuller (ADF)
    para verificar se a série é estacionária.

    Hipótese Nula (H0)
    H0: A série NÃO é estacionária.
         p < 0.05 ->Série é estacionária
    Hipótese Alternativa (H1)
    H1: A série é estacionária.
        p < 0.05 ->Série é estacionári  
    """
    
    resultado = adfuller(serie.dropna())
    
    print("Estatística ADF:", resultado[0])
    print("p-valor:", resultado[1])
    
    if resultado[1] < 0.05:
        print("Conclusão: Série estacionária (rejeita H0)")
    else:
        print("Conclusão: Série NÃO estacionária (não rejeita H0)")


In [4]:
def ajustar_sarima(serie, ordem=(1,0,1), ordem_sazonal=(0,1,1,12)):
    """
    Ajusta um modelo SARIMA à série temporal.
    
    ordem = (p,d,q)
        p = autoregressivo
        d = diferenciação
        q = média móvel
    
    ordem_sazonal = (P,D,Q,s)
        P = AR sazonal
        D = diferenciação sazonal
        Q = MA sazonal
        s = periodicidade (12 para mensal)
    """
    
    modelo = SARIMAX(
        serie,
        order=ordem,
        seasonal_order=ordem_sazonal,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    
    resultado = modelo.fit(disp=False)
    
    print(resultado.summary())
    
    return resultado

In [12]:
def prever_ipca(modelo, passos=48):
    """
    Gera previsão futura do IPCA.
    
    Parâmetros:
    modelo  -> modelo SARIMAX já ajustado
    passos  -> número de meses futuros
    
    Retorna:
    df_previsao com média e intervalos de confiança
    """
    
    previsao = modelo.get_forecast(steps=passos)
    
    media = previsao.predicted_mean
    intervalo = previsao.conf_int()
    
    df_previsao = media.to_frame(name='ipca_previsto')
    df_previsao['limite_inferior'] = intervalo.iloc[:, 0]
    df_previsao['limite_superior'] = intervalo.iloc[:, 1]
    
    return df_previsao

In [10]:
def plotar_previsao(serie, df_previsao):
    """
    Plota série histórica e previsão futura do IPCA com intervalo de confiança.

    Parâmetros:
    serie        -> pandas Series da série histórica (ex: df_ipca['valor'])
    df_previsao  -> DataFrame retornado por prever_ipca() com colunas:
                    'ipca_previsto', 'limite_inferior', 'limite_superior'
    """
    plt.figure(figsize=(12,6))

    # Série histórica
    plt.plot(serie, label='Observado')

    # Previsão média
    plt.plot(df_previsao['ipca_previsto'], label='Previsão', color='orange')

    # Intervalo de confiança
    plt.fill_between(
        df_previsao.index,
        df_previsao['limite_inferior'],
        df_previsao['limite_superior'],
        color='orange',
        alpha=0.3,
        label='Intervalo de confiança'
    )

    plt.legend()
    plt.title('Previsão do IPCA')
    plt.xlabel('Data')
    plt.ylabel('IPCA (%)')
    plt.grid(True)
    plt.show()